# 🚀 AlgorimSeek — Vision + Long-Context Code Model
## Pre-training & Fine-tuning Notebook

**Model:** AlgorimSeek (Based on Qwen2-VL / LFM-3B-VL)

**Capabilities:**
- 📸 Image → Code (read code screenshots)
- 💻 Text → Algorim Code generation
- 🧠 Long-context thinking (DeepSeek-style)
- 🔧 `/compile` `/debug` `/imagine` `/execute` commands
- 🌐 Bilingual: Arabic + English

---
**Run on:** Google Colab (T4/A100) or Kaggle

In [ ]:
# ── 1. INSTALL DEPENDENCIES ──────────────────────────────────
!pip install transformers>=4.40.0 \
             peft>=0.10.0 \
             accelerate>=0.30.0 \
             bitsandbytes>=0.43.0 \
             datasets>=2.18.0 \
             trl>=0.8.6 \
             qwen-vl-utils \
             Pillow \
             wandb \
             flash-attn --no-build-isolation \
             -q
print('✅ Dependencies installed')

In [ ]:
# ── 2. IMPORTS ────────────────────────────────────────────────
import os, json, torch, warnings
from pathlib import Path
from PIL import Image
from datasets import Dataset, load_dataset
from transformers import (
    AutoTokenizer, AutoProcessor,
    Qwen2VLForConditionalGeneration,
    TrainingArguments, Trainer,
    BitsAndBytesConfig, DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

warnings.filterwarnings('ignore')

# ── Device ─────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device}')
if device == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# ── 3. CONFIGURATION ─────────────────────────────────────────
CONFIG = {
    # Model selection (choose one)
    'BASE_MODEL': 'Qwen/Qwen2-VL-7B-Instruct',     # Best for vision+long context
    # 'BASE_MODEL': 'Qwen/Qwen2.5-Coder-7B-Instruct', # Code-focused
    # 'BASE_MODEL': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B', # Thinking model
    # 'BASE_MODEL': 'microsoft/phi-3.5-vision-instruct', # Lightweight vision
    
    'MODEL_NAME': 'AlgorimSeek-7B',
    'OUTPUT_DIR': './algorimseek_output',
    'DATASET_PATH': './datasets/algorim_training.jsonl',
    'VISION_DATASET': './datasets/algorim_vision_training.jsonl',
    
    # Training hyperparameters
    'MAX_SEQ_LENGTH': 4096,     # Long context
    'BATCH_SIZE': 2,
    'GRAD_ACCUM': 8,
    'LEARNING_RATE': 2e-4,
    'NUM_EPOCHS': 3,
    'WARMUP_RATIO': 0.05,
    'SAVE_STEPS': 200,
    
    # LoRA config
    'LORA_R': 64,
    'LORA_ALPHA': 128,
    'LORA_DROPOUT': 0.05,
    'LORA_TARGET': ['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    
    # Quantization (4-bit for VRAM efficiency)
    'USE_4BIT': True,
    'USE_FLASH_ATTN': True,
}

Path(CONFIG['OUTPUT_DIR']).mkdir(exist_ok=True)
print('✅ Configuration set')
print(json.dumps({k:v for k,v in CONFIG.items() if k!='LORA_TARGET'}, indent=2))

In [ ]:
# ── 4. LOAD & PREPARE DATASET ────────────────────────────────

def load_jsonl(path):
    data = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

# System prompt for AlgorimSeek
SYSTEM_PROMPT = """You are AlgorimSeek, an expert AI system for the Algorim programming language (algo a47).

## Algorim Language Rules:
- Functions declared with: `Action name(params): return_type`
- Variable section: `var` ... `begin`
- Assignment operator: `<---`
- Loops: `for (i <--- 0 to n) do` ... `endfor` | `while (cond) do` ... `done`
- Conditionals: `if (cond) then` ... `else` ... `endif`
- Comments: `// text` | Arabic comments supported
- Types: int, float, bool, char, arr, array, string, node
- Return: `action_name <--- value` (assign to function name)

## Supported Commands:
- `/imagine [description]` → Generate code visualization image
- `/compile [code]` → Compile to bytecode (AlgorimVM)
- `/debug [code]` → Step-by-step execution trace
- `/execute [code]` → Run code and show output
- `/explain [code]` → Detailed explanation

## Capabilities:
- Generate Algorim code from text descriptions
- Read code from images (vision)
- Long-context reasoning for complex algorithms
- Bilingual: Arabic and English
- Think step-by-step inside <thinking>...</thinking> tags"""

def format_conversation(entry):
    """Format dataset entry for instruction tuning."""
    messages = entry.get('messages', [])
    
    # Ensure system prompt
    if not messages or messages[0]['role'] != 'system':
        messages = [{'role':'system','content':SYSTEM_PROMPT}] + messages
    else:
        messages[0]['content'] = SYSTEM_PROMPT
    
    return {'messages': messages, 'metadata': entry.get('metadata', {})}

# Load training data
raw_data = load_jsonl(CONFIG['DATASET_PATH'])
print(f'📦 Loaded {len(raw_data)} training examples')

formatted_data = [format_conversation(e) for e in raw_data]

# Split train/eval
split_idx = int(len(formatted_data) * 0.95)
train_data = formatted_data[:split_idx]
eval_data  = formatted_data[split_idx:]

train_dataset = Dataset.from_list(train_data)
eval_dataset  = Dataset.from_list(eval_data)

print(f'✅ Train: {len(train_dataset)} | Eval: {len(eval_dataset)}')

In [ ]:
# ── 5. LOAD BASE MODEL ───────────────────────────────────────

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG['USE_4BIT'],
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
) if CONFIG['USE_4BIT'] else None

print(f'🔄 Loading {CONFIG["BASE_MODEL"]}...')

# Load processor (handles both text + images)
processor = AutoProcessor.from_pretrained(
    CONFIG['BASE_MODEL'],
    trust_remote_code=True
)

# Load model
model = Qwen2VLForConditionalGeneration.from_pretrained(
    CONFIG['BASE_MODEL'],
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
    attn_implementation='flash_attention_2' if CONFIG['USE_FLASH_ATTN'] else 'eager',
    trust_remote_code=True
)

print(f'✅ Model loaded | Params: {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

In [ ]:
# ── 6. APPLY LoRA ────────────────────────────────────────────

if CONFIG['USE_4BIT']:
    model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=CONFIG['LORA_R'],
    lora_alpha=CONFIG['LORA_ALPHA'],
    target_modules=CONFIG['LORA_TARGET'],
    lora_dropout=CONFIG['LORA_DROPOUT'],
    bias='none',
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('✅ LoRA applied')

In [ ]:
# ── 7. TOKENIZATION ──────────────────────────────────────────

def tokenize_fn(examples):
    """Tokenize conversation messages."""
    all_input_ids = []
    all_labels    = []
    all_attn_mask = []
    
    for msgs in examples['messages']:
        # Apply chat template
        text = processor.tokenizer.apply_chat_template(
            msgs,
            tokenize=False,
            add_generation_prompt=False
        )
        tok = processor.tokenizer(
            text,
            max_length=CONFIG['MAX_SEQ_LENGTH'],
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        input_ids = tok['input_ids'][0]
        attn_mask = tok['attention_mask'][0]
        
        # Mask user tokens in labels (only train on assistant responses)
        labels = input_ids.clone()
        # Find assistant token positions (simplified)
        labels[attn_mask == 0] = -100
        
        all_input_ids.append(input_ids.tolist())
        all_labels.append(labels.tolist())
        all_attn_mask.append(attn_mask.tolist())
    
    return {
        'input_ids': all_input_ids,
        'labels': all_labels,
        'attention_mask': all_attn_mask
    }

print('🔄 Tokenizing dataset...')
tokenized_train = train_dataset.map(tokenize_fn, batched=True, batch_size=50, 
                                     remove_columns=['messages','metadata'])
tokenized_eval  = eval_dataset.map(tokenize_fn, batched=True, batch_size=50,
                                    remove_columns=['messages','metadata'])
print('✅ Tokenization complete')
print(f'   Train: {len(tokenized_train)} | Eval: {len(tokenized_eval)}')

In [ ]:
# ── 8. TRAINING ──────────────────────────────────────────────

training_args = TrainingArguments(
    output_dir=CONFIG['OUTPUT_DIR'],
    num_train_epochs=CONFIG['NUM_EPOCHS'],
    per_device_train_batch_size=CONFIG['BATCH_SIZE'],
    per_device_eval_batch_size=CONFIG['BATCH_SIZE'],
    gradient_accumulation_steps=CONFIG['GRAD_ACCUM'],
    learning_rate=CONFIG['LEARNING_RATE'],
    warmup_ratio=CONFIG['WARMUP_RATIO'],
    lr_scheduler_type='cosine',
    fp16=False,
    bf16=True,
    logging_steps=10,
    eval_strategy='steps',
    eval_steps=CONFIG['SAVE_STEPS'],
    save_steps=CONFIG['SAVE_STEPS'],
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    report_to='wandb',    # or 'none'
    run_name=CONFIG['MODEL_NAME'],
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    max_grad_norm=1.0,
    dataloader_num_workers=2,
    group_by_length=True,     # Efficient batching
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
)

print('🚀 Starting training...')
print(f'   Effective batch size: {CONFIG["BATCH_SIZE"] * CONFIG["GRAD_ACCUM"]}')
trainer.train()
print('✅ Training complete!')

In [ ]:
# ── 9. SAVE MODEL ─────────────────────────────────────────────

save_path = f"{CONFIG['OUTPUT_DIR']}/final"
model.save_pretrained(save_path)
processor.save_pretrained(save_path)
print(f'✅ Model saved to {save_path}')

# Save merged model (full weights, no LoRA adapter)
merged_model = model.merge_and_unload()
merged_path  = f"{CONFIG['OUTPUT_DIR']}/merged"
merged_model.save_pretrained(merged_path, safe_serialization=True)
processor.save_pretrained(merged_path)
print(f'✅ Merged model saved to {merged_path}')

# Push to HuggingFace Hub (optional)
# merged_model.push_to_hub('your-username/AlgorimSeek-7B')
# processor.push_to_hub('your-username/AlgorimSeek-7B')

In [ ]:
# ── 10. INFERENCE TEST ───────────────────────────────────────

def generate_algorim(prompt, image_path=None, max_new_tokens=512):
    """Generate Algorim code from text or image+text."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': []}
    ]
    
    # Add image if provided
    if image_path:
        messages[1]['content'].append({'type': 'image', 'image': image_path})
    
    messages[1]['content'].append({'type': 'text', 'text': prompt})
    
    # Format with chat template
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Process inputs
    inputs = processor(text=[text], return_tensors='pt').to(device)
    
    # Generate
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
        )
    
    # Decode
    response = processor.decode(output[0][inputs['input_ids'].shape[1]:], 
                                 skip_special_tokens=True)
    return response

# Test 1: Text → Algorim code
print('\n🧪 Test 1: Text → Code')
print('─' * 50)
result = generate_algorim(
    'Write an Algorim action to compute the height of a binary tree'
)
print(result)

# Test 2: Image → Code
print('\n🧪 Test 2: Image → Code')
print('─' * 50)
result = generate_algorim(
    'Read this code image and explain what the algorithm does',
    image_path='images_generated/demo_render.png'
)
print(result)

# Test 3: /compile command
print('\n🧪 Test 3: /compile command')
print('─' * 50)
result = generate_algorim('/compile factorial(n: int): int - compute n!')
print(result)

---
## 🔭 Vision Dataset Training
Train the model to read code images

In [ ]:
# ── VISION FINE-TUNING ───────────────────────────────────────

def load_vision_dataset(path):
    """Load vision training data with image+text pairs."""
    data = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line.strip())
            img_path = entry.get('image_path', '')
            
            # Skip if image doesn't exist
            if not os.path.exists(img_path):
                continue
            
            messages = [
                {'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': [
                    {'type': 'image', 'image': img_path},
                    {'type': 'text',  'text':  entry.get('prompt', 'What does this code do?')}
                ]},
                {'role': 'assistant', 'content': entry.get('response', '')}
            ]
            data.append({'messages': messages})
    return data

vision_data = load_vision_dataset(CONFIG['VISION_DATASET'])
print(f'👁️ Vision examples: {len(vision_data)}')

vision_dataset = Dataset.from_list(vision_data)

# Use SFTTrainer for vision fine-tuning
sft_config = SFTConfig(
    output_dir=f"{CONFIG['OUTPUT_DIR']}/vision",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=1e-4,
    bf16=True,
    logging_steps=5,
    max_seq_length=2048,
    report_to='none',
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=vision_dataset,
    processing_class=processor,
)

print('🚀 Vision fine-tuning...')
sft_trainer.train()
print('✅ Vision fine-tuning complete!')